# 04 · Exemplar montages

**What it does.** A 5×5 montage of exemplars for each mapped CDI word, so pools can be checked by eye.
- **ImageNet:** tiles cycle **round-robin over the pooled classes** (dog shows 25 breeds); classes are ~1,300 images
  each, so this also matches how the category mean weighs them.
- **LVIS:** tiles are sampled **uniformly over the pool's crops** by default, the mix the category mean averages
  (book has 40,375 crops, phonebook 25). `ROUND_ROBIN = True` shows the pool's breadth instead. Crops are
  letterboxed and captioned with their box size.
- Seeded per word, so montages are reproducible. Words with no class in the dataset are skipped with a note.

**Outputs:** `outputs/montages/{imagenet_val, lvis_uniform, lvis_round_robin}/{valid129,not_valid129}/<word>.jpg`
(`CATEGORIES = None` redraws all, ~30 s).

In [ ]:
# Standard setup shared by every notebook (each notebook is self-contained).
import json
import os
import random
import re
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch, Rectangle
from scipy.cluster.hierarchy import leaves_list, linkage, optimal_leaf_ordering
from scipy.spatial.distance import pdist, squareform
from scipy.stats import false_discovery_control, pearsonr, rankdata, spearmanr, wilcoxon
from IPython.display import Image, Markdown, display

plt.rcParams["font.family"] = ["DejaVu Sans", "sans-serif"]
pd.set_option("display.max_colwidth", 120)


def show_images(paths, width=950):
    """Display saved figures (the files this notebook writes)."""
    for p in map(Path, paths):
        if p.exists():
            display(Markdown(f"`{p.relative_to(ROOT)}`")); display(Image(filename=str(p), width=width))
        else:
            display(Markdown(f"*Missing figure:* `{p}`"))

In [ ]:
# Paths. Run this notebook from analysis/comparison-datasets/notebooks/ (Jupyter's default working directory).
ROOT = Path.cwd().resolve().parent                     # analysis/comparison-datasets
assert (ROOT / "notebooks").is_dir(), "open this notebook from analysis/comparison-datasets/notebooks/"
REPO = ROOT.parents[1]
DATA_DIR = REPO / "data"
MANUSCRIPT_DIR = REPO / "analysis" / "manuscript-2026"
VEDB_DIR = REPO / "analysis" / "vedb-1612"
VOCAB_DIR = DATA_DIR / "vocabulary_lists"
VALID129_TXT = VOCAB_DIR / "included_categories_valid129.txt"

# Inputs written by other pipelines (read only).
EXEMPLAR_EMBED_ROOT = Path(os.environ.get("BV_EXEMPLAR_EMBED_ROOT",
                                          str(MANUSCRIPT_DIR / "exemplar_set_embeddings"))).expanduser()
BV_THINGS_VALID129 = EXEMPLAR_EMBED_ROOT / "valid129"       # manuscript Stage 0: BV + THINGS tables
VEDB_VALID129 = VEDB_DIR / "exemplar_set_embeddings" / "valid129"   # vedb-1612 Stage 0 (files use the "bv_" prefix)

# External data on ccn2.
IMAGENET_DIR = Path("/data2/j7yang/imagenet-1k/extracted")
IMAGENET_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/imagenet_1k/outputs/image_embeddings")
LVIS_ROOT = Path("/data2/j7yang/lvis")
LVIS_CROP_EMBED_ROOT = Path("/ccn2a/dataset/babyview/related_data/lvis/outputs/crop_embeddings")

# Outputs of this folder.
OUT = ROOT / "outputs"
MAPPING_OUT = OUT / "mapping"
MONTAGE_OUT = OUT / "montages"
EXEMPLAR_SETS = OUT / "exemplar_sets"
MODELS = ("clip", "dinov3")

## Parameters

In [ ]:
DATASET = "lvis"
CATEGORIES = ["dog", "book", "toy"]   # None = every mapped CDI word
ROUND_ROBIN = False                   # LVIS only
assert DATASET in ('imagenet', 'lvis'), "DATASET must be one of ('imagenet', 'lvis')"

## Montage layout and sampling (shared)

In [ ]:
from PIL import Image as PILImage, ImageDraw, ImageFont

FONT = "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"
FONT_BOLD = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
N_COLS = 5
N_EXEMPLARS = 25
CELL = 160
CAPTION_H = 16
TITLE_H = 36


def sample_round_robin(wnids, by_wnid, n, rng):
    """Up to n (wnid, filename) pairs, cycling over shuffled classes."""
    pools = {w: rng.sample(by_wnid[w], len(by_wnid[w])) for w in wnids}
    order = rng.sample(list(wnids), len(wnids))
    picks = []
    while len(picks) < n and any(pools.values()):
        for w in order:
            if pools[w] and len(picks) < n:
                picks.append((w, pools[w].pop()))
    return picks


def fit_text(draw, text, font, max_w):
    if draw.textlength(text, font=font) <= max_w:
        return text
    while text and draw.textlength(text + "…", font=font) > max_w:
        text = text[:-1]
    return text + "…"

## ImageNet montages

In [ ]:
def list_images_by_wnid(split_dir: Path) -> dict:
    by_wnid = defaultdict(list)
    with os.scandir(split_dir) as it:
        for e in it:
            if e.name.endswith(".JPEG"):
                by_wnid[e.name[:-5].rsplit("_", 1)[1]].append(e.name)
    for v in by_wnid.values():
        v.sort()
    return by_wnid


def center_square(img: PILImage.Image, size: int) -> PILImage.Image:
    w, h = img.size
    s = min(w, h)
    left, top = (w - s) // 2, (h - s) // 2
    return img.crop((left, top, left + s, top + s)).resize((size, size), PILImage.BICUBIC)


def make_montage(category, picks, labels, n_classes, split_dir, out_path):
    n_rows = (len(picks) + N_COLS - 1) // N_COLS
    W = N_COLS * CELL
    H = TITLE_H + n_rows * (CELL + CAPTION_H)
    canvas = PILImage.new("RGB", (W, H), "white")
    draw = ImageDraw.Draw(canvas)
    title_font = ImageFont.truetype(FONT_BOLD, 20)
    cap_font = ImageFont.truetype(FONT, 11)
    sub = f"{n_classes} ImageNet class{'es' if n_classes != 1 else ''}"
    draw.text((8, 7), category, fill="black", font=title_font)
    tw = draw.textlength(category, font=title_font)
    draw.text((16 + tw, 13), sub, fill="#555555", font=cap_font)

    for i, (wnid, fname) in enumerate(picks):
        r, c = divmod(i, N_COLS)
        x, y = c * CELL, TITLE_H + r * (CELL + CAPTION_H)
        with PILImage.open(split_dir / fname) as im:
            canvas.paste(center_square(im.convert("RGB"), CELL), (x, y))
        cap = fit_text(draw, labels[wnid], cap_font, CELL - 4)
        draw.text((x + 2, y + CELL + 1), cap, fill="#333333", font=cap_font)
    canvas.save(out_path, quality=90)


def imagenet_montages(map_csv: Path = MAPPING_OUT / "imagenet_cdi_hyponym_map.csv", imagenet_dir: Path = IMAGENET_DIR, split: str = "val", out_dir: Path | None = None, categories=None, most_specific_only: bool = False, seed: int = 0):
    """Write a 5x5 montage per mapped CDI word (or the given categories); return the output folder."""

    m = pd.read_csv(map_csv)
    if most_specific_only:
        m = m[m["most_specific"]]
    if categories:
        missing = set(categories) - set(m["cdi_category"])
        if missing:
            raise SystemExit(f"not mapped: {sorted(missing)}")
        m = m[m["cdi_category"].isin(categories)]

    suffix = "_most_specific" if most_specific_only else ""
    out_dir = out_dir or MONTAGE_OUT / f"imagenet_{split}{suffix}"
    valid129 = set(VALID129_TXT.read_text().split())
    for sub in ("valid129", "not_valid129"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)
    split_dir = imagenet_dir / split
    by_wnid = list_images_by_wnid(split_dir)
    labels = dict(zip(m["wnid"], m["imagenet_label"].str.split(", ").str[0]))

    manifest = []
    for category, g in m.groupby("cdi_category"):
        rng = random.Random(f"{seed}:{category}")
        wnids = sorted(g["wnid"])
        picks = sample_round_robin(wnids, by_wnid, N_EXEMPLARS, rng)
        group = "valid129" if category in valid129 else "not_valid129"
        out_path = out_dir / group / f"{category}.jpg"
        make_montage(category, picks, labels, len(wnids), split_dir, out_path)
        manifest += [{"cdi_category": category, "group": group, "tile": i, "wnid": w, "filename": f}
                     for i, (w, f) in enumerate(picks)]
        print(f"{category}: {len(wnids)} classes, {len(picks)} tiles -> {group}/{out_path.name}")
    pd.DataFrame(manifest).to_csv(out_dir / "montage_manifest.csv", index=False)
    print(f"wrote {m['cdi_category'].nunique()} montages to {out_dir}")

    return out_dir

## LVIS montages

In [ ]:
EXCLUSIVE_POOLS = EXEMPLAR_SETS / "lvis_exclusive104" / "lvis_category_classes.csv"


def letterbox(img: PILImage.Image, size: int) -> PILImage.Image:
    """Scale the crop (up or down, aspect kept) to fit the cell, on a gray background."""
    s = size / max(img.size)
    img = img.resize((max(1, round(img.width * s)), max(1, round(img.height * s))), PILImage.BICUBIC)
    canvas = PILImage.new("RGB", (size, size), (235, 235, 235))
    canvas.paste(img, ((size - img.width) // 2, (size - img.height) // 2))
    return canvas


def make_lvis_montage(category, picks, crops, labels, n_classes, out_path):
    n_rows = (len(picks) + N_COLS - 1) // N_COLS
    W, H = N_COLS * CELL, TITLE_H + n_rows * (CELL + CAPTION_H)
    canvas = PILImage.new("RGB", (W, H), "white")
    draw = ImageDraw.Draw(canvas)
    title_font, cap_font = ImageFont.truetype(FONT_BOLD, 20), ImageFont.truetype(FONT, 11)
    draw.text((8, 7), category, fill="black", font=title_font)
    tw = draw.textlength(category, font=title_font)
    draw.text((16 + tw, 13), f"{n_classes} LVIS categor{'ies' if n_classes != 1 else 'y'}",
              fill="#555555", font=cap_font)
    for i, (cat_id, crop_id) in enumerate(picks):
        r, c = divmod(i, N_COLS)
        x, y = c * CELL, TITLE_H + r * (CELL + CAPTION_H)
        row = crops.loc[crop_id]
        with PILImage.open(row["path"]) as im:
            box = im.convert("RGB").crop((row["xmin"], row["ymin"], row["xmax"], row["ymax"]))
        canvas.paste(letterbox(box, CELL), (x, y))
        cap = f"{labels[cat_id]} {round(row['xmax'] - row['xmin'])}x{round(row['ymax'] - row['ymin'])}"
        draw.text((x + 2, y + CELL + 1), fit_text(draw, cap, cap_font, CELL - 4), fill="#333333", font=cap_font)
    canvas.save(out_path, quality=90)


def lvis_montages(map_csv: Path = MAPPING_OUT / "lvis_cdi_hyponym_map.csv", out_dir: Path | None = None, categories=None, exclusive: bool = False, round_robin: bool = False, seed: int = 0):
    """Write a 5x5 LVIS crop montage per mapped CDI word (or the given categories); return the output folder."""

    if exclusive:
        m = pd.read_csv(EXCLUSIVE_POOLS)
    else:
        m = pd.read_csv(map_csv)
    if categories:
        missing = set(categories) - set(m["cdi_category"])
        if missing:
            raise SystemExit(f"not mapped: {sorted(missing)}")
        m = m[m["cdi_category"].isin(categories)]
    name = "lvis_" + ("round_robin" if round_robin else "uniform") + ("_exclusive104" if exclusive else "")
    out_dir = out_dir or MONTAGE_OUT / name
    valid129 = set(VALID129_TXT.read_text().split())
    for sub in ("valid129", "not_valid129"):
        (out_dir / sub).mkdir(parents=True, exist_ok=True)

    manifest = pd.read_csv(MAPPING_OUT / "lvis_crop_manifest.csv").set_index("image_id")
    meta = pd.read_csv(MAPPING_OUT / "lvis_crop_metadata.csv.gz", usecols=["crop_id", "category_id", "lvis_name"])
    # Only crops that could be embedded (>= 1 px each side).
    ok = (manifest["xmax"] - manifest["xmin"] >= 1) & (manifest["ymax"] - manifest["ymin"] >= 1)
    meta = meta[meta["crop_id"].map(ok)]
    by_cat = meta.groupby("category_id")["crop_id"].apply(sorted).to_dict()
    labels = meta.groupby("category_id")["lvis_name"].first().to_dict()

    rows = []
    for category, g in m.groupby("cdi_category"):
        rng = random.Random(f"{seed}:{category}")
        cat_ids = sorted(g["lvis_category_id"])
        if round_robin:
            picks = sample_round_robin(cat_ids, by_cat, N_EXEMPLARS, rng)
        else:
            pool = [(c, k) for c in cat_ids for k in by_cat[c]]
            picks = rng.sample(pool, min(N_EXEMPLARS, len(pool)))
        group = "valid129" if category in valid129 else "not_valid129"
        make_lvis_montage(category, picks, manifest, labels, len(cat_ids), out_dir / group / f"{category}.jpg")
        rows += [{"cdi_category": category, "group": group, "tile": i, "lvis_category_id": c,
                  "lvis_name": labels[c], "crop_id": k} for i, (c, k) in enumerate(picks)]
    pd.DataFrame(rows).to_csv(out_dir / "montage_manifest.csv", index=False)
    print(f"wrote {m['cdi_category'].nunique()} montages to {out_dir}")

    return out_dir

## Run and show

In [ ]:
mapped = set(pd.read_csv(MAPPING_OUT / f"{DATASET}_cdi_hyponym_map.csv").cdi_category)
if CATEGORIES:
    skipped = [w for w in CATEGORIES if w not in mapped]
    if skipped:
        print(f"no {DATASET} class for {skipped}; skipped")
    cats = [w for w in CATEGORIES if w in mapped]
else:
    cats = None
out_dir = imagenet_montages(categories=cats) if DATASET == "imagenet" else lvis_montages(categories=cats, round_robin=ROUND_ROBIN)
show_images([next(out_dir.glob(f"*/{w}.jpg")) for w in (cats or ["dog"])], width=520)

### Review checklist

- Is each tile the labeled object (LVIS boxes are converted from x, y, w, h)?
- Do the pools look like what a child calls the word?
- LVIS: how many tiles are too small to recognize?